# Model Karşılaştırma — Tüm Modeller
**Plant Leaf Disease Classification Dataset**  
Bu notebook tüm 7 modelin sonuçlarını karşılaştırır ve akademik rapor için tablo/grafik üretir.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

results_dir = Path("./results")

result_files = [
    "cnn_baseline_results.json",
    "vit_results.json",
    "vit_seg_results.json",
    "swin_results.json",
    "swin_seg_results.json",
    "botanix_cnn_scratch_results.json",
    "botanix_cnn_scratch_seg_results.json",
]

records = []
for fname in result_files:
    fpath = results_dir / fname
    if fpath.exists():
        with open(fpath) as f:
            records.append(json.load(f))
    else:
        print(f"Bulunamadı: {fname}")

df = pd.DataFrame([{
    "Model":            r["model"],
    "Accuracy":         r["accuracy"],
    "Precision":        r["precision"],
    "Recall":           r["recall"],
    "F1 Score":         r["f1_score"],
    "ROC AUC":          r.get("roc_auc_macro", float("nan")),
    "Train Time (min)": r["training_time_min"],
    "Infer (ms/img)":   r["inference_time_ms_per_image"],
    "Params (M)":       round(r["total_params"] / 1e6, 1),
} for r in records])

df = df.sort_values("Accuracy", ascending=False).reset_index(drop=True)
print(df.to_string(index=False))

In [ ]:
# ── Metrik Karşılaştırma Grafiği ─────────────────────────────────────────────
metrics = ["Accuracy", "Precision", "Recall", "F1 Score", "ROC AUC"]
x = np.arange(len(df))
width = 0.15

fig, ax = plt.subplots(figsize=(18, 7))
colors = ["#2196F3", "#4CAF50", "#FF9800", "#E91E63", "#9C27B0"]

for i, (metric, color) in enumerate(zip(metrics, colors)):
    bars = ax.bar(x + i * width, df[metric], width, label=metric, color=color, alpha=0.85)
    for bar in bars:
        h = bar.get_height()
        if not np.isnan(h):
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.002,
                    f"{h:.3f}", ha='center', va='bottom', fontsize=7)

ax.set_xlabel("Model")
ax.set_ylabel("Score")
ax.set_title("Model Karşılaştırması — Tüm Metrikler")
ax.set_xticks(x + width * 2)
ax.set_xticklabels(df["Model"], rotation=20, ha="right", fontsize=9)
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("./results/model_comparison_metrics.png", dpi=150)
plt.show()

In [ ]:
# ── Eğitim Süresi vs Accuracy ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(df["Model"], df["Train Time (min)"], color="#5C6BC0", alpha=0.85)
axes[0].set_xlabel("Eğitim Süresi (dakika)")
axes[0].set_title("Eğitim Süresi")
axes[0].grid(axis='x', alpha=0.3)

scatter = axes[1].scatter(
    df["Train Time (min)"], df["Accuracy"],
    s=df["Params (M)"] * 3, alpha=0.8,
    c=range(len(df)), cmap="tab10"
)
for _, row in df.iterrows():
    axes[1].annotate(row["Model"][:15],
                     (row["Train Time (min)"], row["Accuracy"]),
                     textcoords="offset points", xytext=(5, 5), fontsize=8)
axes[1].set_xlabel("Eğitim Süresi (dakika)")
axes[1].set_ylabel("Test Accuracy")
axes[1].set_title("Accuracy vs Eğitim Süresi\n(Nokta boyutu = Parametre sayısı)")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("./results/model_comparison_efficiency.png", dpi=150)
plt.show()

In [ ]:
# ── Akademik Tablo (LaTeX formatı) ───────────────────────────────────────────
print("=== LaTeX Tablosu ===")
print(df[["Model", "Accuracy", "Precision", "Recall", "F1 Score", "ROC AUC",
          "Train Time (min)", "Infer (ms/img)", "Params (M)"]]
      .to_latex(index=False, float_format="{:.4f}".format))

In [ ]:
# ── Özet Rapor ────────────────────────────────────────────────────────────────
best = df.iloc[0]
print(f"\n{'='*60}")
print(f"EN İYİ MODEL: {best['Model']}")
print(f"  Accuracy:  {best['Accuracy']:.4f}")
print(f"  F1 Score:  {best['F1 Score']:.4f}")
print(f"  ROC AUC:   {best['ROC AUC']:.4f}")
print(f"  Eğitim:    {best['Train Time (min)']:.1f} dakika")
print(f"  Inference: {best['Infer (ms/img)']:.2f} ms/görüntü")
print(f"{'='*60}")

print(f"\nSegmentation etkisi (Accuracy farkı):")
pairs = [
    ("Vision Transformer (ViT-B/16)",        "ViT + Segmentation"),
    ("Swin Transformer (Swin-B)",            "Swin Transformer + Segmentation"),
    ("Botanix CNN from Scratch",             "Botanix CNN from Scratch + Segmentation"),
]
for base_name, seg_name in pairs:
    base_row = df[df["Model"] == base_name]
    seg_row  = df[df["Model"] == seg_name]
    if not base_row.empty and not seg_row.empty:
        diff = seg_row["Accuracy"].values[0] - base_row["Accuracy"].values[0]
        sign = "+" if diff >= 0 else ""
        print(f"  {base_name[:28]:<28} → Seg: {sign}{diff:.4f}")

print(f"\nTransfer Learning etkisi (Accuracy farkı):")
tl_pairs = [
    ("CNN Baseline", "Botanix CNN from Scratch"),
]
for pretrain_name, scratch_name in tl_pairs:
    pt_row = df[df["Model"] == pretrain_name]
    sc_row = df[df["Model"] == scratch_name]
    if not pt_row.empty and not sc_row.empty:
        diff = pt_row["Accuracy"].values[0] - sc_row["Accuracy"].values[0]
        sign = "+" if diff >= 0 else ""
        print(f"  Pretrained CNN vs Scratch CNN: {sign}{diff:.4f} (pretrained lehine)")

# ── Radar/Spider Grafiği ─────────────────────────────────────────────────────
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

radar_metrics = ["Accuracy", "Precision", "Recall", "F1 Score", "ROC AUC"]
N = len(radar_metrics)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
colors_r = plt.cm.tab10(np.linspace(0, 1, len(df)))

for idx, (_, row) in enumerate(df.iterrows()):
    values = [row[m] if not np.isnan(row[m]) else 0 for m in radar_metrics]
    values += values[:1]
    ax.plot(angles, values, "o-", lw=2, label=row["Model"][:20], color=colors_r[idx])
    ax.fill(angles, values, alpha=0.07, color=colors_r[idx])

ax.set_thetagrids(np.degrees(angles[:-1]), radar_metrics, fontsize=11)
ax.set_ylim(0, 1)
ax.set_title("Model Karşılaştırması — Radar Grafiği", size=14, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=9)
plt.tight_layout()
plt.savefig("./results/model_comparison_radar.png", dpi=150)
plt.show()

print("\nTüm karşılaştırma grafikleri ./results/ dizinine kaydedildi.")